# LILA BLIP Settings Notebook

Sample captions from the full LILA train split, then summarize the recurring background, weather, lighting, and season settings.

This notebook also serializes the underrepresented classes selected from `lila-suppl-data.csv` so the augmentation notebook and the later SpeciesNet fine-tuning workflow use the same class list.

In [ ]:
from pathlib import Path
import pandas as pd

from lila_pipeline_helpers import (
    load_lila_suppl_data,
    select_bottom_percent_classes,
    build_class_targets,
    serialize_class_targets,
    load_lila_train_split,
    sample_class_images,
    write_manifest,
)

ROOT = Path.cwd()
ARTIFACT_DIR = ROOT / 'artifacts' / 'lila_blip_settings'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
suppl = load_lila_suppl_data(ROOT / 'lila-suppl-data.csv')
selected = select_bottom_percent_classes(suppl, percentile=5.0)
targets = build_class_targets(selected)
paths = serialize_class_targets(targets, ARTIFACT_DIR)
selected[['common_name', 'count_train', 'threshold_count', 'deficit_to_threshold', 'target_added']].to_csv(ARTIFACT_DIR / 'selected_bottom5_summary.csv', index=False)
print(paths)
print(targets[['common_name', 'count_train', 'target_added']].to_string(index=False))


In [ ]:
train_df = load_lila_train_split(ROOT / 'lila_splits' / 'lila_splits_train.csv')
train_df = train_df[train_df['common_name'].isin(targets['common_name'])].reset_index(drop=True)
print(train_df[['common_name']].value_counts().head())


In [ ]:
# Sample captions from each selected class.
# Replace the captioning call with the BLIP model you prefer; this cell is structured
# so you can run it on the whole LILA train set or on per-class samples.
caption_manifest_rows = []
for class_name in targets['common_name']:
    sample_df = sample_class_images(train_df, class_name, n=min(10, len(train_df[train_df['common_name'] == class_name])), seed=42)
    for i, row in sample_df.iterrows():
        caption_manifest_rows.append({
            'common_name': class_name,
            'gs_filepath': row['gs_filepath'],
            'project': row.get('project', ''),
            'location': row.get('location', ''),
            'caption': '',
            'background_setting': '',
            'weather_setting': '',
            'lighting_setting': '',
            'season_setting': '',
        })
caption_manifest = pd.DataFrame(caption_manifest_rows)
write_manifest(caption_manifest, ARTIFACT_DIR / 'blip_caption_samples_manifest.csv')
caption_manifest.head()


## Next step
Run BLIP over `caption_manifest`, then summarize the sampled captions into reusable setting banks for background, weather, lighting, and season.